**#PyTorch Mini Project (Dataset-Independent)**
Choose One Dataset

Students must choose one dataset from the following list and build a PyTorch ANN model.

1️⃣ MNIST Handwritten Digit Dataset

Digits 0–9 classification.

Dataset link:
https://www.kaggle.com/datasets/hojjatk/mnist-dataset

2️⃣ Fashion-MNIST Dataset

Clothing image classification.

Dataset link:
https://www.kaggle.com/datasets/zalando-research/fashionmnist

3️⃣ Iris Dataset

Flower species classification.

Dataset link:
https://www.kaggle.com/datasets/uciml/iris

4️⃣ Breast Cancer Wisconsin Dataset

Binary classification (malignant vs benign).

Dataset link:
https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

5️⃣ Titanic Survival Dataset

Predict whether a passenger survived.

Dataset link:
https://www.kaggle.com/datasets/yasserh/titanic-dataset

# Topics Students Must Use

Students must apply the following PyTorch concepts in their project:

Why PyTorch

Tensors

Autograd

PyTorch Training Pipeline

torch.nn Module & torch.optim Module

Dataset and DataLoader

Building ANN using PyTorch

Model training and evaluation



In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader



# Question 1

Load the dataset you selected and perform basic preprocessing.

Explain:

number of samples

number of features

target variable

train-test split

# Write answer of Question 1

In [3]:
df = pd.read_csv('Iris.csv')
df = df.drop(columns=['Id'])

df.head()

print("Number of samples:", df.shape[0])
print("Number of features:", df.shape[1] - 1)
print("Target variable classes:", df['Species'].unique())

X = df.drop(columns=['Species'])
y = df['Species']

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Number of samples: 150
Number of features: 4
Target variable classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
Train samples: 120
Test samples: 30


# Question 2

Convert the dataset features into PyTorch tensors.

Explain:

what a tensor is

tensor shape of your dataset

why tensors are required in PyTorch

#Write answer of Question 2

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X_train_tensor shape:", X_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([120, 4])
X_test_tensor shape: torch.Size([30, 4])
y_train_tensor shape: torch.Size([120])
y_test_tensor shape: torch.Size([30])


# Question 3

Create a simple example using Autograd:

define a tensor with requires_grad=True

perform a mathematical operation

compute gradient using .backward()

Explain why autograd is important for neural networks.

#Write answer of Question 3

In [5]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y.backward()

print("x:", x.item())
print("y:", y.item())
print("Gradient dy dx:", x.grad.item())


### Explanation:-
# requires_grad = True: Tells PyTorch to remember this tensor, so it can calculate its gradient later.
# .backward(): Automatically calculates the gradient (derivative) — no manual math needed.
# Why autograd matters for neural networks:
# Neural networks learn by adjusting weights to reduce error. To know how much to adjust each weight, we need gradients. With thousands of weights, calculating this by hand is impossible. Autograd does it automatically — this is the core of how neural networks learn (backpropagation).

x: 3.0
y: 16.0
Gradient dy dx: 8.0


# Question 4

Create a custom Dataset class and use DataLoader.

Explain the role of:

Dataset

DataLoader

batch_size

shuffle

#  Write answer of Question 4

In [6]:
class IrisDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

train_dataset = IrisDataset(X_train_tensor, y_train_tensor)
test_dataset = IrisDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

for features, labels in train_loader:
    print("Batch features shape:", features.shape)
    print("Batch labels shape:", labels.shape)
    break


### Dataset: A class that stores our data and tells PyTorch how to get one sample at a time (__getitem__) and how many samples exist (__len__).
### DataLoader: Takes the Dataset and automatically creates batches, shuffles data, and feeds it to the model during training. Saves us from writing manual loops.
### batch_size: Number of samples processed together in one step (here, 16). Smaller batches = more updates, slower but sometimes better learning.
### shuffle: Randomly mixes data order each epoch. We use shuffle=True for training (avoids the model memorizing order) and shuffle=False for testing (order doesn't matter there).

Batch features shape: torch.Size([16, 4])
Batch labels shape: torch.Size([16])


# Question 5

Design an Artificial Neural Network (ANN) using torch.nn.Module.

Your model must include:

input layer

at least one hidden layer

output layer

Explain the purpose of:

nn.Linear

activation functions

loss function

# Write answer of Question 5

In [7]:
class ANN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

input_size = 4
hidden_size = 8
output_size = 3

model = ANN(input_size, hidden_size, output_size)
print(model)

criterion = nn.CrossEntropyLoss()

### Explanation:-
## nn.Linear: Creates a fully connected layer. It takes input numbers, multiplies them by weights, adds a bias, and passes the result forward. This is the basic building block of a neural network.
## Activation functions (like ReLU): Add non-linearity to the model. Without them, the network would just be doing simple math (like linear regression), no matter how many layers it has. Activation functions let the network learn complex patterns.
## Loss function (CrossEntropyLoss): Measures how wrong the model's predictions are compared to actual labels. Since this is a multi-class classification problem (3 flower types), CrossEntropyLoss is the right choice. The training goal is to minimize this loss.

ANN(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)


# Question 6

Implement a complete PyTorch training pipeline that includes:

Forward pass

Loss calculation

Backpropagation

Optimizer step

Clearing gradients

Train the model for multiple epochs.

#  Write answer of Question 6

In [8]:
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):
    total_loss = 0
    for features, labels in train_loader:

        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

Epoch [10/100], Loss: 0.3143
Epoch [20/100], Loss: 0.1533
Epoch [30/100], Loss: 0.0928
Epoch [40/100], Loss: 0.0744
Epoch [50/100], Loss: 0.0545
Epoch [60/100], Loss: 0.0479
Epoch [70/100], Loss: 0.0446
Epoch [80/100], Loss: 0.0443
Epoch [90/100], Loss: 0.0485
Epoch [100/100], Loss: 0.0397


# Question 7

Evaluate your model on the test dataset and compute:

accuracy

loss

Explain whether the model is underfitting or overfitting.

#  Write answer of Question 7

In [11]:
model.eval()

correct = 0
total = 0
test_loss = 0

with torch.no_grad():
    for features, labels in test_loader:
        outputs = model(features)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
avg_test_loss = test_loss / len(test_loader)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Loss: 0.0906
Test Accuracy: 96.67%


# Question 8

Modify the architecture and compare performance:

Example modifications:

increase hidden neurons

add another hidden layer

change activation function

Explain the effect on performance.

#  Write answer of Question 8

In [12]:
class ANN_v2(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(ANN_v2, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.tanh = nn.Tanh()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.tanh(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

# Create the new model
model_v2 = ANN_v2(input_size=4, hidden_size1=16, hidden_size2=8, output_size=3)
print(model_v2)

# Train model_v2
optimizer_v2 = optim.Adam(model_v2.parameters(), lr=0.01)
criterion_v2 = nn.CrossEntropyLoss()

epochs = 100
for epoch in range(epochs):
    total_loss = 0
    for features, labels in train_loader:
        outputs = model_v2(features)
        loss = criterion_v2(outputs, labels)
        loss.backward()
        optimizer_v2.step()
        optimizer_v2.zero_grad()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluate model_v2
model_v2.eval()
correct = 0
total = 0
test_loss = 0
with torch.no_grad():
    for features, labels in test_loader:
        outputs = model_v2(features)
        loss = criterion_v2(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Model v2 Test Loss: {test_loss/len(test_loader):.4f}")
print(f"Model v2 Test Accuracy: {correct/total*100:.2f}%")

ANN_v2(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (tanh): Tanh()
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (relu): ReLU()
  (fc3): Linear(in_features=8, out_features=3, bias=True)
)
Epoch [10/100], Loss: 0.0939
Epoch [20/100], Loss: 0.0649
Epoch [30/100], Loss: 0.0406
Epoch [40/100], Loss: 0.0424
Epoch [50/100], Loss: 0.0369
Epoch [60/100], Loss: 0.0467
Epoch [70/100], Loss: 0.0316
Epoch [80/100], Loss: 0.0460
Epoch [90/100], Loss: 0.0301
Epoch [100/100], Loss: 0.0306
Model v2 Test Loss: 0.1306
Model v2 Test Accuracy: 96.67%
